# 04 — Regime Gate

Train an interpretable gate using only reduced temperature and reduced density, then evaluate the deployable soft-gate model on natural and stress held-out test sets.

In [ ]:
!pip install -q pysr scikit-learn

In [ ]:
from pathlib import Path
import json
import pickle
import time

import numpy as np
import pandas as pd


def find_project_root():
    current = Path.cwd()
    candidates = [current, current.parent]

    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate
        if (candidate / "notebooks").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
MODEL_DIR = RESULTS_DIR / "models"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Processed data directory not found: {DATA_DIR}. "
        "Run 01_CO2_Data_Generation first."
    )

TABLE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

train_df = pd.read_csv(DATA_DIR / "co2_train.csv")
validation_df = pd.read_csv(DATA_DIR / "co2_validation.csv")
test_df = pd.read_csv(DATA_DIR / "co2_test.csv")
stress_validation_df = pd.read_csv(
    DATA_DIR / "co2_stress_validation.csv"
)
stress_test_df = pd.read_csv(DATA_DIR / "co2_stress_test.csv")

FEATURE_COLUMNS = ["T_reduced", "rho_reduced"]
TARGET_COLUMN = "delta_Z"

with open(MODEL_DIR / "global_symbolic_model.pkl", "rb") as file:
    global_model = pickle.load(file)

with open(MODEL_DIR / "global_selection.json", "r") as file:
    selected_global_index = json.load(file)["selected_global_index"]

selected_global_index = int(selected_global_index)

with open(MODEL_DIR / "final_regime_models.pkl", "rb") as file:
    final_regime_models = pickle.load(file)

with open(MODEL_DIR / "final_regime_indices.json", "r") as file:
    final_regime_indices = {
        name: int(index)
        for name, index in json.load(file).items()
    }

REGIME_NAMES = [
    "near_ideal",
    "attraction_dominated",
    "excluded_volume_dominated"
]


In [ ]:
from pysr import PySRRegressor


In [ ]:
def evaluate_predictions(
    dataset,
    predicted_delta_z,
    dataset_name,
    model_name
):
    true_delta_z = dataset["delta_Z"].to_numpy()
    ideal_pressure = dataset["p_ideal_Pa"].to_numpy()
    true_pressure = dataset["p_real_Pa"].to_numpy()

    predicted_pressure = (
        ideal_pressure
        * (1.0 + predicted_delta_z)
    )

    delta_error = predicted_delta_z - true_delta_z
    pressure_error = (
        np.abs(predicted_pressure - true_pressure)
        / np.abs(true_pressure)
        * 100.0
    )

    return {
        "model": model_name,
        "dataset": dataset_name,
        "delta_Z_RMSE": np.sqrt(np.mean(delta_error ** 2)),
        "delta_Z_MAE": np.mean(np.abs(delta_error)),
        "pressure_MAPE_percent": np.mean(pressure_error),
        "pressure_max_error_percent": np.max(pressure_error)
    }


In [ ]:
def predict_selected_global(
    model,
    selected_index,
    dataset
):
    X = dataset[
        FEATURE_COLUMNS
    ].to_numpy()

    return model.predict(
        X,
        index=selected_index
    )


def predict_selected_regime_models(
    models,
    selected_indices,
    dataset
):
    predictions = np.empty(
        len(dataset),
        dtype=float
    )

    for regime_name in REGIME_NAMES:
        mask = (
            dataset["regime"].to_numpy()
            == regime_name
        )

        if not np.any(mask):
            continue

        X = dataset.loc[
            mask,
            FEATURE_COLUMNS
        ].to_numpy()

        predictions[mask] = (
            models[regime_name].predict(
                X,
                index=selected_indices[
                    regime_name
                ]
            )
        )

    return predictions

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix
)

GATE_FEATURES = [
    "T_reduced",
    "rho_reduced"
]

gate_candidates = {}

for max_depth in [
    3,
    4,
    5,
    6,
    8,
    10,
    12
]:
    gate = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=47
    )

    gate.fit(
        train_df[GATE_FEATURES],
        train_df["regime"]
    )

    gate_candidates[max_depth] = gate

In [ ]:
def predict_with_regime_gate(
    gate,
    dataset
):
    predicted_regimes = gate.predict(
        dataset[GATE_FEATURES]
    )

    predictions = np.empty(
        len(dataset),
        dtype=float
    )

    for regime_name in REGIME_NAMES:
        mask = (
            predicted_regimes
            == regime_name
        )

        if not np.any(mask):
            continue

        X = dataset.loc[
            mask,
            FEATURE_COLUMNS
        ].to_numpy()

        predictions[mask] = (
            final_regime_models[
                regime_name
            ].predict(
                X,
                index=final_regime_indices[
                    regime_name
                ]
            )
        )

    return predictions, predicted_regimes

In [ ]:
gate_results = []

for max_depth, gate in gate_candidates.items():
    for dataset, dataset_name in [
        (
            validation_df,
            "natural_validation"
        ),
        (
            stress_validation_df,
            "stress_validation"
        )
    ]:
        predictions, predicted_regimes = (
            predict_with_regime_gate(
                gate,
                dataset
            )
        )

        metrics = evaluate_predictions(
            dataset,
            predictions,
            dataset_name,
            f"gate_depth_{max_depth}"
        )

        metrics["max_depth"] = max_depth

        metrics["accuracy"] = accuracy_score(
            dataset["regime"],
            predicted_regimes
        )

        metrics["balanced_accuracy"] = (
            balanced_accuracy_score(
                dataset["regime"],
                predicted_regimes
            )
        )

        gate_results.append(metrics)

gate_results = pd.DataFrame(
    gate_results
)

gate_results[
    [
        "max_depth",
        "dataset",
        "accuracy",
        "balanced_accuracy",
        "delta_Z_RMSE",
        "pressure_MAPE_percent",
        "pressure_max_error_percent"
    ]
]

In [ ]:
gate_depth_summary = (
    gate_results
    .groupby("max_depth")
    .agg(
        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean"
        ),
        mean_delta_Z_RMSE=(
            "delta_Z_RMSE",
            "mean"
        ),
        mean_pressure_MAPE=(
            "pressure_MAPE_percent",
            "mean"
        ),
        worst_pressure_error=(
            "pressure_max_error_percent",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        [
            "mean_pressure_MAPE",
            "worst_pressure_error"
        ]
    )
)

gate_depth_summary

In [ ]:
FINAL_GATE_DEPTH = 10

final_gate = gate_candidates[
    FINAL_GATE_DEPTH
]

print(
    "Selected gate depth:",
    FINAL_GATE_DEPTH
)

In [ ]:
for dataset, dataset_name in [
    (
        validation_df,
        "natural_validation"
    ),
    (
        stress_validation_df,
        "stress_validation"
    )
]:
    predicted_regimes = final_gate.predict(
        dataset[GATE_FEATURES]
    )

    confusion = pd.DataFrame(
        confusion_matrix(
            dataset["regime"],
            predicted_regimes,
            labels=REGIME_NAMES
        ),
        index=[
            "true_" + name
            for name in REGIME_NAMES
        ],
        columns=[
            "pred_" + name
            for name in REGIME_NAMES
        ]
    )

    print("\n", dataset_name)
    display(confusion)

In [ ]:
def create_gate_error_table(
    gate,
    dataset,
    dataset_name
):
    result = dataset.copy().reset_index(
        drop=True
    )

    predictions, predicted_regimes = (
        predict_with_regime_gate(
            gate,
            result
        )
    )

    true_pressure = result[
        "p_real_Pa"
    ].to_numpy()

    predicted_pressure = (
        result["p_ideal_Pa"].to_numpy()
        * (1.0 + predictions)
    )

    result["dataset"] = dataset_name
    result["predicted_regime"] = (
        predicted_regimes
    )
    result["gate_correct"] = (
        result["regime"]
        == result["predicted_regime"]
    )
    result["gate_delta_Z_prediction"] = (
        predictions
    )
    result["gate_pressure_error_percent"] = (
        np.abs(
            predicted_pressure - true_pressure
        )
        / np.abs(true_pressure)
        * 100.0
    )

    return result


natural_gate_errors = create_gate_error_table(
    final_gate,
    validation_df,
    "natural_validation"
)

stress_gate_errors = create_gate_error_table(
    final_gate,
    stress_validation_df,
    "stress_validation"
)

In [ ]:
worst_gate_points = (
    stress_gate_errors
    .sort_values(
        "gate_pressure_error_percent",
        ascending=False
    )
    .head(20)
)

worst_gate_points[
    [
        "T_K",
        "rho_mol_m3",
        "T_reduced",
        "rho_reduced",
        "delta_Z",
        "regime",
        "predicted_regime",
        "gate_correct",
        "gate_delta_Z_prediction",
        "gate_pressure_error_percent"
    ]
]

In [ ]:
gate_correctness_summary = pd.concat([
    natural_gate_errors,
    stress_gate_errors
]).groupby(
    [
        "dataset",
        "gate_correct"
    ]
).agg(
    count=(
        "gate_pressure_error_percent",
        "size"
    ),
    mean_error=(
        "gate_pressure_error_percent",
        "mean"
    ),
    error_95th=(
        "gate_pressure_error_percent",
        lambda x: np.percentile(x, 95)
    ),
    max_error=(
        "gate_pressure_error_percent",
        "max"
    )
).reset_index()

gate_correctness_summary

In [ ]:
def predict_with_soft_regime_gate(
    gate,
    dataset
):
    X_gate = dataset[GATE_FEATURES]

    regime_probabilities = gate.predict_proba(
        X_gate
    )

    all_predictions = np.zeros(
        (
            len(dataset),
            len(REGIME_NAMES)
        ),
        dtype=float
    )

    for regime_index, regime_name in enumerate(
        REGIME_NAMES
    ):
        X = dataset[
            FEATURE_COLUMNS
        ].to_numpy()

        all_predictions[:, regime_index] = (
            final_regime_models[
                regime_name
            ].predict(
                X,
                index=final_regime_indices[
                    regime_name
                ]
            )
        )

    probability_columns = []

    for regime_name in REGIME_NAMES:
        class_index = list(
            gate.classes_
        ).index(regime_name)

        probability_columns.append(
            regime_probabilities[:, class_index]
        )

    ordered_probabilities = np.column_stack(
        probability_columns
    )

    blended_predictions = np.sum(
        ordered_probabilities
        * all_predictions,
        axis=1
    )

    return (
        blended_predictions,
        ordered_probabilities
    )

In [ ]:
soft_gate_results = []

for dataset, dataset_name in [
    (
        validation_df,
        "natural_validation"
    ),
    (
        stress_validation_df,
        "stress_validation"
    )
]:
    hard_predictions, _ = (
        predict_with_regime_gate(
            final_gate,
            dataset
        )
    )

    soft_predictions, _ = (
        predict_with_soft_regime_gate(
            final_gate,
            dataset
        )
    )

    soft_gate_results.append(
        evaluate_predictions(
            dataset,
            hard_predictions,
            dataset_name,
            "hard_gate"
        )
    )

    soft_gate_results.append(
        evaluate_predictions(
            dataset,
            soft_predictions,
            dataset_name,
            "soft_gate"
        )
    )

soft_gate_comparison = pd.DataFrame(
    soft_gate_results
)

soft_gate_comparison[
    [
        "model",
        "dataset",
        "delta_Z_RMSE",
        "delta_Z_MAE",
        "pressure_MAPE_percent",
        "pressure_max_error_percent"
    ]
]

In [ ]:
soft_gate_comparison[
    [
        "model",
        "dataset",
        "pressure_MAPE_percent",
        "pressure_max_error_percent"
    ]
].sort_values(
    [
        "dataset",
        "pressure_MAPE_percent"
    ]
)

In [ ]:
FINAL_ROUTING_METHOD = "soft_gate"
FINAL_GATE_DEPTH = 10

def predict_final_model(dataset):
    predictions, probabilities = (
        predict_with_soft_regime_gate(
            final_gate,
            dataset
        )
    )

    return predictions


final_validation_results = []

for dataset, dataset_name in [
    (
        validation_df,
        "natural_validation"
    ),
    (
        stress_validation_df,
        "stress_validation"
    )
]:
    predictions = predict_final_model(
        dataset
    )

    final_validation_results.append(
        evaluate_predictions(
            dataset,
            predictions,
            dataset_name,
            "final_soft_gate_model"
        )
    )

final_validation_results = pd.DataFrame(
    final_validation_results
)

final_validation_results

In [ ]:
final_test_results = []

for dataset, dataset_name in [
    (
        test_df,
        "natural_test"
    ),
    (
        stress_test_df,
        "stress_test"
    )
]:
    predictions = predict_final_model(
        dataset
    )

    final_test_results.append(
        evaluate_predictions(
            dataset,
            predictions,
            dataset_name,
            "final_soft_gate_model"
        )
    )

final_test_results = pd.DataFrame(
    final_test_results
)

final_test_results

In [ ]:
test_comparison_rows = []

for dataset, dataset_name in [
    (
        test_df,
        "natural_test"
    ),
    (
        stress_test_df,
        "stress_test"
    )
]:
    X = dataset[
        FEATURE_COLUMNS
    ].to_numpy()

    global_predictions = (
        global_model.predict(
            X,
            index=selected_global_index
        )
    )

    oracle_predictions = (
        predict_selected_regime_models(
            final_regime_models,
            final_regime_indices,
            dataset
        )
    )

    hard_predictions, _ = (
        predict_with_regime_gate(
            final_gate,
            dataset
        )
    )

    soft_predictions = predict_final_model(
        dataset
    )

    test_comparison_rows.extend([
        evaluate_predictions(
            dataset,
            global_predictions,
            dataset_name,
            "global_symbolic"
        ),
        evaluate_predictions(
            dataset,
            oracle_predictions,
            dataset_name,
            "oracle_regime_specific"
        ),
        evaluate_predictions(
            dataset,
            hard_predictions,
            dataset_name,
            "hard_gate"
        ),
        evaluate_predictions(
            dataset,
            soft_predictions,
            dataset_name,
            "final_soft_gate_model"
        )
    ])

test_comparison = pd.DataFrame(
    test_comparison_rows
)

test_comparison[
    [
        "model",
        "dataset",
        "delta_Z_RMSE",
        "delta_Z_MAE",
        "pressure_MAPE_percent",
        "pressure_max_error_percent"
    ]
]

In [ ]:
final_validation_and_test = pd.concat([
    final_validation_results,
    final_test_results
], ignore_index=True)

final_validation_and_test.to_csv(
    TABLE_DIR / "final_validation_and_test_results.csv",
    index=False
)

test_comparison.to_csv(
    TABLE_DIR / "final_test_comparison.csv",
    index=False
)

regime_equations_final = pd.DataFrame([
    {
        "regime": regime_name,
        "equation_index": final_regime_indices[regime_name],
        "equation": str(
            final_regime_models[regime_name].sympy(
                index=final_regime_indices[regime_name]
            )
        )
    }
    for regime_name in REGIME_NAMES
])

regime_equations_final.to_csv(
    TABLE_DIR / "final_regime_equations.csv",
    index=False
)

gate_results.to_csv(
    TABLE_DIR / "gate_results.csv",
    index=False
)

gate_depth_summary.to_csv(
    TABLE_DIR / "gate_depth_summary.csv",
    index=False
)

soft_gate_comparison.to_csv(
    TABLE_DIR / "soft_gate_comparison.csv",
    index=False
)

gate_correctness_summary.to_csv(
    TABLE_DIR / "gate_correctness_summary.csv",
    index=False
)

print("Final results saved to:", TABLE_DIR)


In [ ]:
with open(MODEL_DIR / "final_gate.pkl", "wb") as file:
    pickle.dump(final_gate, file)

with open(MODEL_DIR / "gate_config.json", "w") as file:
    json.dump(
        {
            "final_gate_depth": int(FINAL_GATE_DEPTH),
            "routing_method": FINAL_ROUTING_METHOD,
            "gate_features": GATE_FEATURES,
            "regime_names": REGIME_NAMES
        },
        file,
        indent=2
    )

print("Final gate saved to:", MODEL_DIR)
